In [1]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 50.2 MB/s eta 0:00:00


In [2]:
import google.generativeai as genai


In [3]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

In [4]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

test_items= [
 "The earth is spherical.",
 "I like to eat at a restaurant.", 
 "I love watching football.",
"I love playing football with friends",
 "I love going out with friends"]
    
query = "i played football"

response = genai.embed_content(model='models/text-embedding-004',
                               content=test_items,
                               task_type='retrieval_document')
response_query = genai.embed_content(model='models/text-embedding-004',
                               content=query,
                               task_type='retrieval_document')


In [5]:
res_embedding = response['embedding']
query_embedding = response_query['embedding']



In [6]:
#Naive nearset neighbour 
n_neighbors=2
nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='brute').fit(res_embedding) 
naive_distances, naive_indices = nbrs.kneighbors(np.expand_dims(query_embedding, axis = 0))

print(naive_distances)
print(naive_indices)

[[0.47917184 0.52068319]]
[[3 2]]


In [7]:
#algorithm- ball_tree due to high dimensional vectors or kd_tree otherwise
nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='ball_tree').fit(res_embedding) 
distances, indices = nbrs.kneighbors(np.expand_dims(query_embedding, axis = 0))

print(distances)
print(naive_indices)

[[0.47917184 0.52068319]]
[[3 2]]


In [8]:
import faiss
M=32 
d=768 
index = faiss.IndexHNSWFlat(d, M)
index.add(np.array(res_embedding)) #build the index using the embeddings in Snippet 9
#execute the ANN search
index.search(np.expand_dims(query_embedding, axis=0), k=2)

(array([[0.22960567, 0.27111098]], dtype=float32), array([[3, 2]]))